<a href="https://colab.research.google.com/github/lopezjuana/my-first-blog/blob/master/regresion_validacion_gridsearch_explicada.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Regresión Supervisada con Validación Cruzada y Ajuste de Hiperparámetros


# Introducción

En esta práctica vamos a trabajar un flujo completo de Machine Learning para un problema de **regresión supervisada**.

La idea es adaptar conceptos que suelen verse en clasificación hacia un escenario de regresión.

## Temas principales

- Tratamiento de valores faltantes.
- Variables categóricas.
- Pipelines.
- Validación cruzada.
- Ajuste de hiperparámetros.
- Evaluación de modelos de regresión.

Trabajaremos utilizando Scikit-Learn y un dataset real.


## Importación de librerías

In [8]:
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import KFold
from sklearn.model_selection import GridSearchCV

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import StandardScaler

from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression

from sklearn.metrics import mean_absolute_error
from sklearn.metrics import mean_squared_error
from sklearn.metrics import r2_score



# Dataset

Utilizaremos el dataset **California Housing**.

Este dataset contiene información sobre viviendas en California y el objetivo será predecir el valor medio de las viviendas (`MedHouseVal`).

## ¿Por qué este dataset?

Porque:

- Es relativamente pequeño.
- Tiene variables fáciles de interpretar.
- Funciona muy bien para regresión.
- Es ideal para prácticas académicas.


In [9]:
from sklearn.datasets import fetch_california_housing

housing = fetch_california_housing(as_frame=True)

df = housing.frame.copy()

df.head()


,MedInc,HouseAge,AveRooms,AveBedrms,Population,AveOccup,Latitude,Longitude,MedHouseVal
0,8.3252,41.0,6.984127,1.023810,322.0,2.555556,37.88,-122.23,4.526
1,8.3014,21.0,6.238137,0.971880,2401.0,2.109842,37.86,-122.22,3.585
2,7.2574,52.0,8.288136,1.073446,496.0,2.802260,37.85,-122.24,3.521
3,5.6431,52.0,5.817352,1.073059,558.0,2.547945,37.85,-122.25,3.413
4,3.8462,52.0,6.281853,1.081081,565.0,2.181467,37.85,-122.25,3.422


## Exploración inicial del dataset


Antes de entrenar modelos es importante entender:

- Cantidad de filas y columnas.
- Tipo de variables.
- Posibles valores faltantes.
- Distribución general de los datos.


In [10]:
print(df.shape)


(20640, 9)


In [11]:
df.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20640 entries, 0 to 20639
Data columns (total 9 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   MedInc       20640 non-null  float64
 1   HouseAge     20640 non-null  float64
 2   AveRooms     20640 non-null  float64
 3   AveBedrms    20640 non-null  float64
 4   Population   20640 non-null  float64
 5   AveOccup     20640 non-null  float64
 6   Latitude     20640 non-null  float64
 7   Longitude    20640 non-null  float64
 8   MedHouseVal  20640 non-null  float64
dtypes: float64(9)
memory usage: 1.4 MB


In [12]:
df.describe()


,MedInc,HouseAge,AveRooms,AveBedrms,Population,AveOccup,Latitude,Longitude,MedHouseVal
count,20640.000000,20640.000000,20640.000000,20640.000000,20640.000000,20640.000000,20640.000000,20640.000000,20640.000000
mean,3.870671,28.639486,5.429000,1.096675,1425.476744,3.070655,35.631861,-119.569704,2.068558
std,1.899822,12.585558,2.474173,0.473911,1132.462122,10.386050,2.135952,2.003532,1.153956
min,0.499900,1.000000,0.846154,0.333333,3.000000,0.692308,32.540000,-124.350000,0.149990
25%,2.563400,18.000000,4.440716,1.006079,787.000000,2.429741,33.930000,-121.800000,1.196000
50%,3.534800,29.000000,5.229129,1.048780,1166.000000,2.818116,34.260000,-118.490000,1.797000
75%,4.743250,37.000000,6.052381,1.099526,1725.000000,3.282261,37.710000,-118.010000,2.647250
max,15.000100,52.000000,141.909091,34.066667,35682.000000,1243.333333,41.950000,-114.310000,5.000010



# Creación de variable categórica

El dataset original tiene solamente variables numéricas.

Para poder practicar:
- imputación categórica,
- codificación,
- pipelines mixtos,

vamos a crear artificialmente una variable categórica.


In [13]:
df['income_level'] = pd.cut(
    df['MedInc'],
    bins=[0, 2, 4, 6, np.inf],
    labels=['bajo', 'medio', 'alto', 'muy_alto']
)

df[['MedInc', 'income_level']].head()


,MedInc,income_level
0,8.3252,muy_alto
1,8.3014,muy_alto
2,7.2574,muy_alto
3,5.6431,alto
4,3.8462,medio



# Generación de valores faltantes

En problemas reales es muy común tener datos incompletos.

Para practicar imputación vamos a generar artificialmente:

- valores faltantes numéricos,
- valores faltantes categóricos.


In [14]:
np.random.seed(42)

missing_num = np.random.choice(df.index, size=500, replace=False)
df.loc[missing_num, 'HouseAge'] = np.nan

missing_cat = np.random.choice(df.index, size=300, replace=False)
df.loc[missing_cat, 'income_level'] = np.nan

df.isnull().sum()


,0
MedInc,0
HouseAge,500
AveRooms,0
AveBedrms,0
Population,0
AveOccup,0
Latitude,0
Longitude,0
MedHouseVal,0
income_level,300



# Variables predictoras y variable objetivo

## Variable objetivo

La variable objetivo será:

- `MedHouseVal`

Es decir, el valor medio de las viviendas.

## Variables predictoras

Serán todas las demás columnas.


In [15]:
X = df.drop('MedHouseVal', axis=1)
y = df['MedHouseVal']



# División entrenamiento y prueba

Separar entrenamiento y prueba permite:

- entrenar el modelo con una parte de los datos,
- evaluar generalización con datos no vistos.


In [16]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)



# Identificación de variables numéricas y categóricas

Necesitamos separar tipos de columnas porque cada tipo tendrá un preprocesamiento distinto.


In [17]:
numeric_features = X.select_dtypes(include=['int64', 'float64']).columns
categorical_features = X.select_dtypes(include=['object', 'category']).columns

print('Numéricas:', numeric_features)
print('Categóricas:', categorical_features)


Numéricas: Index(['MedInc', 'HouseAge', 'AveRooms', 'AveBedrms', 'Population', 'AveOccup',
       'Latitude', 'Longitude'],
      dtype='object')
Categóricas: Index(['income_level'], dtype='object')



# Preprocesamiento

## Variables numéricas

Aplicaremos:

- imputación con media,
- escalado.

## Variables categóricas

Aplicaremos:

- imputación con moda,
- One Hot Encoding.


In [18]:
numeric_transformer = Pipeline(
    steps=[
        ('imputer', SimpleImputer(strategy='mean')),
        ('scaler', StandardScaler())
    ]
)

categorical_transformer = Pipeline(
    steps=[
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('onehot', OneHotEncoder(handle_unknown='ignore'))
    ]
)



# ColumnTransformer

`ColumnTransformer` permite aplicar distintos procesos según el tipo de columna.


In [19]:
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features)
    ]
)



# Modelo base: Regresión Lineal

Comenzaremos con un modelo sencillo y muy interpretable.


In [21]:
pipeline_lr = Pipeline(
    steps=[
        ('preprocessor', preprocessor),
        ('model', LinearRegression())
    ]
)

pipeline_lr.fit(X_train, y_train)


Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer()),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  Index(['MedInc', 'HouseAge', 'AveRooms', 'AveBedrms', 'Population', 'AveOccup',
       'Latitude', 'Longitude'],
      dtype='object')),
                                                 ('cat',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('onehot',
                                                                   OneHotEncoder(handle_unknown='ignore'))]),
                                                  Index(['income_level'], dtype='object'))])),
                ('model', LinearRegression())])

In [25]:
modelo = pipeline_lr.named_steps['model']
print(modelo.coef_)
print(modelo.intercept_)

[ 0.69994593  0.12719273 -0.31707175  0.35886147 -0.00118398 -0.04103831
 -0.88555559 -0.8600869  -0.00767199 -0.2456148  -0.16349208  0.41677888]
2.136215162908981


Obtener nombres reales de columnas transformadas

In [29]:
feature_names = pipeline_lr.named_steps[
    'preprocessor'
].get_feature_names_out()
coeficientes = pd.DataFrame({
    'Variable': feature_names,
    'Coeficiente': modelo.coef_
})
#Ordenalos por valor absoluto de coeficiente
coeficientes['AbsCoef'] = coeficientes['Coeficiente'].abs()

coeficientes_ordenados = coeficientes.sort_values(
    by='AbsCoef',
    ascending=False
)
coeficientes_ordenados

,Variable,Coeficiente,AbsCoef
6,num__Latitude,-0.885556,0.885556
7,num__Longitude,-0.860087,0.860087
0,num__MedInc,0.699946,0.699946
11,cat__income_level_muy_alto,0.416779,0.416779
3,num__AveBedrms,0.358861,0.358861
2,num__AveRooms,-0.317072,0.317072
9,cat__income_level_bajo,-0.245615,0.245615
10,cat__income_level_medio,-0.163492,0.163492
1,num__HouseAge,0.127193,0.127193
5,num__AveOccup,-0.041038,0.041038


**Ejercicio de interpretación**  
Consigna  
a) ¿Qué variable tiene mayor impacto positivo?  
b) ¿Cuál tiene impacto negativo?  
c) ¿Qué significa un coeficiente cercano a 0?  
d) ¿Qué variables parecen más importantes?


# Predicciones y métricas

Utilizaremos:

- MAE (error absoluto medio),
- RMSE (raíz del error cuadrático medio),
- R².


In [30]:
y_pred = pipeline_lr.predict(X_test)

mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print('MAE:', mae)
print('RMSE:', rmse)
print('R2:', r2)


MAE: 0.5323583929042487
RMSE: 0.7457711505057839
R2: 0.5755717351133136



# Validación cruzada

La validación cruzada permite obtener una medida más robusta del rendimiento del modelo.

En este caso utilizaremos:

- 5 folds,
- métrica R².


In [ ]:
cv = KFold(n_splits=5, shuffle=True, random_state=42)

scores = cross_val_score(
    pipeline_lr,
    X,
    y,
    cv=cv,
    scoring='r2'
)

print('Scores R2:', scores)
print('Promedio:', scores.mean())
print('Desvío estándar:', scores.std())



# Modelo más complejo: Random Forest

Ahora probaremos un modelo más potente basado en múltiples árboles de decisión.


In [31]:
pipeline_rf = Pipeline(
    steps=[
        ('preprocessor', preprocessor),
        ('model', RandomForestRegressor(random_state=42))
    ]
)



# Ajuste de hiperparámetros con GridSearchCV

`GridSearchCV` prueba múltiples combinaciones de hiperparámetros para encontrar la mejor configuración posible.


In [32]:
param_grid = {
    'model__n_estimators': [50, 100],
    'model__max_depth': [5, 10, None],
    'model__min_samples_leaf': [1, 3, 5]
}

grid_search = GridSearchCV(
    pipeline_rf,
    param_grid,
    cv=5,
    scoring='r2',
    n_jobs=-1
)

grid_search.fit(X_train, y_train)


GridSearchCV(cv=5,
             estimator=Pipeline(steps=[('preprocessor',
                                        ColumnTransformer(transformers=[('num',
                                                                         Pipeline(steps=[('imputer',
                                                                                          SimpleImputer()),
                                                                                         ('scaler',
                                                                                          StandardScaler())]),
                                                                         Index(['MedInc', 'HouseAge', 'AveRooms', 'AveBedrms', 'Population', 'AveOccup',
       'Latitude', 'Longitude'],
      dtype='object')),
                                                                        ('cat',
                                                                         Pipeline(steps=[('imputer',
                                                                                          SimpleImputer(strategy='most_frequent')),
                                                                                         ('onehot',
                                                                                          OneHotEncoder(handle_unknown='ignore'))]),
                                                                         Index(['income_level'], dtype='object'))])),
                                       ('model',
                                        RandomForestRegressor(random_state=42))]),
             n_jobs=-1,
             param_grid={'model__max_depth': [5, 10, None],
                         'model__min_samples_leaf': [1, 3, 5],
                         'model__n_estimators': [50, 100]},
             scoring='r2')

## Mejores hiperparámetros encontrados

In [33]:
print('Mejores parámetros:')
print(grid_search.best_params_)

print('\nMejor score:')
print(grid_search.best_score_)


Mejores parámetros:
{'model__max_depth': None, 'model__min_samples_leaf': 1, 'model__n_estimators': 100}

Mejor score:
0.8043948984079574



# Evaluación final del mejor modelo

Una vez encontrada la mejor combinación de hiperparámetros evaluamos el modelo sobre el conjunto de prueba.


In [34]:
best_model = grid_search.best_estimator_

y_pred_best = best_model.predict(X_test)

mae = mean_absolute_error(y_test, y_pred_best)
rmse = np.sqrt(mean_squared_error(y_test, y_pred_best))
r2 = r2_score(y_test, y_pred_best)

print('MAE:', mae)
print('RMSE:', rmse)
print('R2:', r2)


MAE: 0.3313556526889536
RMSE: 0.5106741193910732
R2: 0.8009872791272146



# Comparación de modelos

Compararemos el desempeño de:

- Regresión Lineal,
- Random Forest.


In [ ]:
resultados = pd.DataFrame({
    'Modelo': ['Regresión Lineal', 'Random Forest'],
    'R2': [
        r2_score(y_test, y_pred),
        r2_score(y_test, y_pred_best)
    ]
})

resultados



# Visualización

Graficaremos valores reales vs predicciones.


In [ ]:
plt.figure(figsize=(8,6))

plt.scatter(y_test, y_pred_best, alpha=0.5)

plt.xlabel('Valores reales')
plt.ylabel('Predicciones')
plt.title('Valores reales vs predichos')

plt.show()



# Conclusiones

## En esta práctica vimos:

- Tratamiento de datos faltantes.
- Variables categóricas.
- Pipelines.
- Validación cruzada.
- Ajuste de hiperparámetros.
- Evaluación de modelos de regresión.

## Posibles extensiones

- Gradient Boosting.
- XGBoost.
- RandomizedSearchCV.
- Importancia de variables.
- Feature Engineering.
